# Trabalho Final — Modelagem Funcional Geral (Etapa 3)

## Detecção de Uso de EPI em Ambiente Simples

**Disciplina:** ESZA019 — Visão Computacional — UFABC 2026.2
**Professor:** Celso Setsuo Kurashima
**Equipe:** Grupo 2

### Integrantes da equipe
- Lucas Rodrigues Teixeira — RA 11202131394
- Pedro Henrique Garcez Silva — RA 11202130642
- Roberto Sene Azevedo — RA 11202020360

**Entregável:** Etapa 3 — Modelagem Funcional Geral (esboço)
**Data de publicação:** 15/07/2026

> **Escopo deste documento.** Esta é a entrega da **Etapa 3** do Trabalho Final: a **Modelagem
> Funcional Geral** do sistema. Trata-se de um documento de **projeto** (*design*), que define
> **requisitos técnicos, especificações, funções e funcionalidades, diagramas de blocos de
> implementação, o método de calibração e o método de avaliação funcional**. Ainda **não** contém
> resultados experimentais nem os testes com voluntários — estes serão produzidos nas etapas
> seguintes (Desenvolvimento, Etapa 4, e Testes, Etapas 5–7).


## Sumário

1. [Introdução, contexto e cenário de aplicação](#1-introdução-contexto-e-cenário-de-aplicação)
2. [Objetivo da modelagem funcional](#2-objetivo-da-modelagem-funcional)
3. [Requisitos técnicos e especificações](#3-requisitos-técnicos-e-especificações)
   - 3.1 [Delimitação do escopo — "ambiente simples"](#31-delimitação-do-escopo--ambiente-simples)
   - 3.2 [Requisitos funcionais (RF)](#32-requisitos-funcionais-rf)
   - 3.3 [Requisitos não funcionais (RNF)](#33-requisitos-não-funcionais-rnf)
   - 3.4 [Especificações de hardware e software](#34-especificações-de-hardware-e-software)
4. [Funções e funcionalidades do sistema](#4-funções-e-funcionalidades-do-sistema)
5. [Arquitetura e diagramas de blocos de implementação](#5-arquitetura-e-diagramas-de-blocos-de-implementação)
   - 5.1 [Diagrama de blocos geral (pipeline)](#51-diagrama-de-blocos-geral-pipeline)
   - 5.2 [Detalhamento dos blocos](#52-detalhamento-dos-blocos)
   - 5.3 [Fluxograma da lógica de decisão](#53-fluxograma-da-lógica-de-decisão)
   - 5.4 [Esboço de implementação (protótipo)](#54-esboço-de-implementação-protótipo)
6. [Método de calibração](#6-método-de-calibração)
7. [Método de avaliação funcional](#7-método-de-avaliação-funcional)
8. [Cronograma e próximas etapas](#8-cronograma-e-próximas-etapas)
9. [Declaração de uso de Inteligência Artificial Generativa](#9-declaração-de-uso-de-inteligência-artificial-generativa)
10. [Referências](#10-referências)


## 1. Introdução, contexto e cenário de aplicação

Equipamentos de Proteção Individual (**EPI**) — capacete, colete refletivo (alta visibilidade), óculos
de proteção, luvas — são a primeira barreira contra acidentes em canteiros de obra, oficinas e
laboratórios. A fiscalização do **uso correto** desses equipamentos, porém, ainda é feita quase sempre
de forma **manual e intermitente** por um responsável de segurança, o que é sujeito a falhas humanas,
não cobre todos os momentos e não gera registro objetivo.

A partir da fase de empatia do Trabalho Final (Etapas 1 e 2), a equipe definiu como **cenário de
aplicação** o **monitoramento automático do uso de EPI na entrada de uma área controlada** (por exemplo,
o acesso a uma oficina ou laboratório): uma câmera posicionada na entrada verifica, em **tempo real**, se
a pessoa que se aproxima está usando os EPIs obrigatórios e sinaliza **conforme / não conforme**.

Para manter o projeto **específico e bem delimitado** — conforme recomenda o item 8 do manual ("menos é
mais, desde que seja preciso") — o sistema é modelado para um **ambiente simples**: uma pessoa por vez,
iluminação controlada, fundo relativamente neutro e câmera fixa. Essa delimitação é o que viabiliza uma
solução baseada em **técnicas clássicas de Visão Computacional** (detecção e rastreamento), sem
necessidade de redes neurais, e ainda assim precisa e mensurável.

> **Observação sobre as entrevistas empáticas.** As entrevistas que fundamentam a escolha do tema, sua
> justificativa e originalidade foram conduzidas e relatadas na **Etapa 2 (Tema do Trabalho)**. Este
> documento de Etapa 3 concentra-se na **modelagem funcional** e apenas as referencia; os registros das
> entrevistas permanecem no entregável da Etapa 2.


## 2. Objetivo da modelagem funcional

Esta etapa tem por objetivo **especificar de forma completa como o sistema funcionará**, antes de sua
implementação. Concretamente, o documento define:

- os **requisitos técnicos** e as **especificações** do sistema (o que ele deve fazer e sob quais
  condições);
- as **funções e funcionalidades** (decomposição do sistema em capacidades);
- os **diagramas de blocos de implementação** (como os módulos se conectam e qual o fluxo dos dados);
- o **método de calibração** da câmera (requisito obrigatório do trabalho);
- o **método de avaliação funcional** (as métricas objetivas que comprovarão a robustez da solução).

A **trilha técnica escolhida** (Requisito B do manual) é a **Detecção e Rastreamento por métodos
clássicos**: detecção de pessoa, definição de regiões de interesse (cabeça e tronco), verificação da
presença dos EPIs por **segmentação de cor** e **análise de forma**, e **rastreamento temporal** para
estabilizar a decisão e tratar oclusões.


## 3. Requisitos técnicos e especificações

### 3.1 Delimitação do escopo — "ambiente simples"

O sistema é projetado para as seguintes condições controladas, que caracterizam o "ambiente simples":

| Condição | Especificação de projeto |
|----------|--------------------------|
| Número de pessoas | **Uma pessoa por vez** no campo de visão (fila/eclusa de entrada). |
| Iluminação | Interna, **difusa e estável** (sem contraluz forte nem sombras duras). |
| Fundo | Relativamente **neutro e estático** (parede/piso de cor uniforme). |
| Câmera | **Fixa** (tripé/suporte), previamente **calibrada**. |
| Distância | Pessoa a ~1,5–3 m da câmera, aproximadamente frontal. |
| EPIs alvo | **Capacete** e **colete de alta visibilidade** (cores saturadas — detecção clássica por cor). |

**Justificativa da delimitação.** EPIs como capacete e colete de alta visibilidade são projetados
justamente para serem **visualmente conspícuos** (cores saturadas: amarelo, laranja, branco, azul). Isso
os torna bons candidatos à detecção por **segmentação de cor no espaço HSV**, dispensando redes neurais e
mantendo o sistema leve e explicável — adequado ao objetivo pedagógico do trabalho.


### 3.2 Requisitos funcionais (RF)

| ID | Requisito funcional |
|----|---------------------|
| RF-01 | Capturar vídeo em **tempo real** de uma webcam/câmera USB. |
| RF-02 | Aplicar a **calibração** previamente obtida para corrigir a distorção de cada quadro. |
| RF-03 | **Detectar a presença de uma pessoa** no quadro e delimitar sua *bounding box*. |
| RF-04 | Definir automaticamente as **regiões de interesse (ROIs)** de cabeça e tronco a partir da caixa da pessoa. |
| RF-05 | Verificar a presença de **capacete** na ROI da cabeça (segmentação de cor + forma). |
| RF-06 | Verificar a presença de **colete de alta visibilidade** na ROI do tronco (segmentação de cor). |
| RF-07 | **Rastrear** a pessoa entre quadros para estabilizar a decisão e tratar oclusões parciais. |
| RF-08 | **Classificar** o estado como **CONFORME** (todos os EPIs presentes) ou **NÃO CONFORME**. |
| RF-09 | **Exibir** o resultado sobreposto ao vídeo (caixas, rótulos e cores de status) e emitir **alerta** visual. |
| RF-10 | **Registrar** métricas de execução (FPS/latência) e permitir a coleta de dados para validação. |


### 3.3 Requisitos não funcionais (RNF)

| ID | Requisito não funcional | Meta de projeto |
|----|--------------------------|-----------------|
| RNF-01 | **Desempenho em tempo real** | ≥ 15 FPS em CPU comum (sem GPU). |
| RNF-02 | **Latência** de decisão por quadro | ≤ 66 ms/quadro. |
| RNF-03 | **Robustez temporal** | Decisão confirmada por *N* quadros consecutivos (evita "piscar"). |
| RNF-04 | **Portabilidade** | Rodar em Linux com Python 3 e OpenCV, sem *hardware* dedicado. |
| RNF-05 | **Explicabilidade** | Cada decisão deve ser rastreável (qual ROI/cor motivou o rótulo). |
| RNF-06 | **Usabilidade** | Interface visual clara (verde = conforme, vermelho = não conforme). |


### 3.4 Especificações de hardware e software

**Hardware.**
- Webcam USB (≥ 640×480, 30 FPS) ou câmera do notebook.
- Tripé/suporte para manter a câmera **fixa** (a calibração exige que a câmera não se mova).
- Tabuleiro de xadrez impresso (padrão de calibração, cantos internos conhecidos).
- Computador comum (CPU), sem necessidade de GPU.

**Software.**
- **Python 3** + **OpenCV 4.x** (`opencv-python`), **NumPy**.
- Módulos previstos: `aquisicao`, `calibracao`, `deteccao_pessoa`, `analise_epi`, `rastreamento`,
  `decisao`, `interface`, `metricas`.
- Reaproveitamento direto dos **Laboratórios 4 e 5** (calibração de câmera) da própria equipe.


## 4. Funções e funcionalidades do sistema

O sistema é decomposto nas seguintes **funções** (capacidades internas) e **funcionalidades**
(o que o usuário percebe):

**Funções (internas).**
1. **Aquisição de vídeo** — leitura contínua de quadros da câmera.
2. **Correção geométrica** — *undistort* de cada quadro com os parâmetros de calibração.
3. **Detecção de pessoa** — localização da pessoa no quadro (descritor **HOG + SVM** de pedestres do
   OpenCV, ou subtração de fundo quando a câmera é fixa).
4. **Extração de ROIs** — cálculo das sub-regiões de **cabeça** (faixa superior da caixa) e **tronco**
   (faixa central) a partir da *bounding box* da pessoa.
5. **Análise de EPI por cor/forma** — no espaço **HSV**, segmentação das cores típicas de capacete
   (na ROI da cabeça) e de colete de alta visibilidade (na ROI do tronco), seguida de operações
   morfológicas e verificação de **área/forma** dos *blobs*.
6. **Rastreamento** — associação da pessoa entre quadros (rastreador por centróide ou `cv2.TrackerKCF`/
   `CSRT`), com histórico para **decisão temporal** e tratamento de **oclusões**.
7. **Decisão e classificação** — combinação lógica das evidências (capacete ∧ colete) em
   **CONFORME / NÃO CONFORME**, confirmada por *N* quadros.
8. **Medição de desempenho** — cálculo de **FPS/latência** e exportação de dados para validação.

**Funcionalidades (percebidas pelo usuário).**
- Vídeo ao vivo com **caixa da pessoa**, **ROIs** e **rótulos** ("Capacete: OK", "Colete: ausente").
- **Status global** destacado (faixa verde/vermelha) e **alerta** quando não conforme.
- **Contador de FPS** na tela e opção de **gravar** a sessão para análise posterior.


## 5. Arquitetura e diagramas de blocos de implementação

### 5.1 Diagrama de blocos geral (pipeline)

O fluxo de dados percorre os blocos abaixo, do sensor à decisão:

```
   [ Câmera USB ]
        |
        v
+------------------+     +---------------------+     +----------------------+
|  (1) Aquisição   | --> |  (2) Calibração /   | --> |  (3) Pré-processamento|
|  de vídeo (BGR)  |     |  undistort (K,dist) |     |  (BGR->HSV, blur)     |
+------------------+     +---------------------+     +----------+-----------+
                                                                |
                                                                v
                                                   +--------------------------+
                                                   |  (4) Detecção de pessoa  |
                                                   |  (HOG+SVM / subtr. fundo)|
                                                   +------------+-------------+
                                                                |  bounding box
                                                                v
                                +---------------------------------------------------+
                                |            (5) Extração de ROIs                    |
                                |   ROI cabeça (topo ~25%)   ROI tronco (~25%-65%)   |
                                +----------------+---------------------+-------------+
                                                 |                     |
                                                 v                     v
                                   +----------------------+  +------------------------+
                                   | (6a) Análise capacete|  | (6b) Análise colete    |
                                   |  HSV + morfologia +  |  |  HSV (alta visib.) +   |
                                   |  forma/area          |  |  area minima           |
                                   +----------+-----------+  +-----------+------------+
                                              |                          |
                                              +-----------+--------------+
                                                          v
                                             +----------------------------+
                                             | (7) Rastreamento + decisao |
                                             | temporal (N quadros)       |
                                             +-------------+--------------+
                                                           v
                                         +-------------------------------------+
                                         | (8) Saida: rotulos + status +       |
                                         |     alerta + metricas (FPS)         |
                                         +-------------------------------------+
```


### 5.2 Detalhamento dos blocos

| Bloco | Entrada | Processamento | Saída |
|-------|---------|---------------|-------|
| (1) Aquisição | Câmera USB | `cv2.VideoCapture` lê o quadro | Quadro BGR |
| (2) Calibração | Quadro BGR + `K`, `dist` | `cv2.undistort` (parâmetros do Lab 4) | Quadro sem distorção |
| (3) Pré-proc. | Quadro corrigido | Conversão BGR→HSV, leve *blur* | Imagem HSV |
| (4) Detecção de pessoa | Quadro corrigido | HOG+SVM (`detectMultiScale`) ou `createBackgroundSubtractorMOG2` | *Bounding box* da pessoa |
| (5) Extração de ROIs | *Bounding box* | Recorte proporcional: cabeça (topo) e tronco (centro) | ROI cabeça, ROI tronco |
| (6a) Capacete | ROI cabeça (HSV) | Máscara de cor + `morphologyEx` + área/forma do *blob* | Booleano "capacete presente" |
| (6b) Colete | ROI tronco (HSV) | Máscara de cor de alta visibilidade + área mínima | Booleano "colete presente" |
| (7) Rastreamento/decisão | Detecções por quadro | Associação por centróide + confirmação em *N* quadros | Estado estável por pessoa |
| (8) Saída | Estado + evidências | Desenho de caixas/rótulos, alerta, FPS | Vídeo anotado + *log* |

**Nota de projeto (ordem calibração × processamento).** Conforme consolidado no Laboratório 1, qualquer
processamento sobre a imagem ocorre **depois da captura/leitura do quadro** e **antes da exibição ou
gravação**. Por isso a **calibração (undistort)** é o **primeiro** processamento do *pipeline*: todos os
blocos seguintes trabalham sobre a imagem já corrigida.


### 5.3 Fluxograma da lógica de decisão

```
             +----------------------------+
             |  Pessoa detectada no quadro?|
             +-------------+--------------+
                           | não
                 +---------+---------+
                 |                   v
                 | sim         [ aguarda / sem status ]
                 v
      +---------------------------+
      | Capacete presente na ROI  |
      | da cabeca?                |
      +------+-------------+------+
             | sim         | nao
             v             v
   +------------------+   [ NAO CONFORME: falta capacete ]
   | Colete presente  |
   | na ROI do tronco?|
   +----+--------+----+
        | sim    | nao
        v        v
 [ CONFORME ]   [ NAO CONFORME: falta colete ]
        |
        v
 confirma em N quadros consecutivos -> emite status estavel + alerta se NAO CONFORME
```


### 5.4 Esboço de implementação (protótipo)

Os trechos abaixo são um **esboço** (não a versão final) que ilustra como cada bloco será implementado
com técnicas **clássicas**. Servem para fixar as interfaces entre os módulos; os valores de cor, limiares
e proporções de ROI serão **ajustados experimentalmente** na Etapa 4.


In [ ]:
import cv2
import numpy as np

# --- Bloco (2): calibracao previamente obtida (Lab 4). Valores de exemplo. ---
K = np.array([[692.56, 0., 320.24],
              [0., 689.08, 247.95],
              [0., 0., 1.]])
dist = np.array([[-0.0325, 0.4864, 0.0025, -0.0097, -1.2327]])

# --- Bloco (4): detector de pessoa classico (HOG + SVM de pedestres). ---
hog = cv2.HOGDescriptor()
hog.setSVMDetector(cv2.HOGDescriptor_getDefaultPeopleDetector())

# --- Bloco (6): faixas de cor (HSV) para capacete e colete de alta visibilidade. ---
# OBS: valores iniciais; serao CALIBRADOS na Etapa 4 com os EPIs reais.
FAIXAS_CAPACETE = {                    # capacete: cores saturadas comuns
    "amarelo": ((20, 100, 100), (35, 255, 255)),
    "branco":  ((0, 0, 200),   (180, 40, 255)),
    "azul":    ((100, 120, 80), (130, 255, 255)),
}
FAIXA_COLETE = ((15, 120, 120), (40, 255, 255))   # laranja/amarelo alta visibilidade

def presenca_por_cor(roi_bgr, faixas, area_min_frac=0.06):
    """Retorna True se ha area de cor suficiente na ROI (evidencia do EPI)."""
    if roi_bgr.size == 0:
        return False, 0.0
    hsv = cv2.cvtColor(roi_bgr, cv2.COLOR_BGR2HSV)
    faixas = faixas if isinstance(faixas, list) else [faixas]
    mask_total = np.zeros(hsv.shape[:2], np.uint8)
    for lo, hi in faixas:
        mask_total |= cv2.inRange(hsv, np.array(lo), np.array(hi))
    # limpeza morfologica (remove ruido, preenche buracos)
    mask_total = cv2.morphologyEx(mask_total, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    frac = cv2.countNonZero(mask_total) / float(mask_total.size)
    return frac >= area_min_frac, frac


In [ ]:
def analisar_pessoa(frame_bgr, box):
    """Bloco (5)+(6): dada a caixa da pessoa, extrai ROIs e verifica EPIs."""
    x, y, w, h = box
    # ROI da cabeca = faixa superior (~25%); ROI do tronco = faixa central (25%-65%).
    roi_cabeca = frame_bgr[y:y + int(0.25*h), x:x + w]
    roi_tronco = frame_bgr[y + int(0.25*h):y + int(0.65*h), x:x + w]

    faixas_cap = list(FAIXAS_CAPACETE.values())
    tem_capacete, _ = presenca_por_cor(roi_cabeca, faixas_cap, area_min_frac=0.08)
    tem_colete,  _ = presenca_por_cor(roi_tronco, FAIXA_COLETE, area_min_frac=0.10)

    conforme = tem_capacete and tem_colete          # Bloco (7)/(8): decisao
    return {"capacete": tem_capacete, "colete": tem_colete, "conforme": conforme}

# --- Laco principal (esboco): aquisicao -> undistort -> deteccao -> analise -> saida ---
# cap = cv2.VideoCapture(0)
# while True:
#     ok, frame = cap.read()
#     if not ok: break
#     frame = cv2.undistort(frame, K, dist)                       # (2)
#     caixas, _ = hog.detectMultiScale(frame, winStride=(8, 8))   # (4)
#     for box in caixas:
#         r = analisar_pessoa(frame, box)                          # (5)+(6)+(7)
#         cor = (0, 200, 0) if r["conforme"] else (0, 0, 255)
#         x, y, w, h = box
#         cv2.rectangle(frame, (x, y), (x+w, y+h), cor, 2)         # (8)
#     cv2.imshow("Deteccao de EPI", frame)
#     if cv2.waitKey(1) == ord('q'): break
# cap.release(); cv2.destroyAllWindows()


## 6. Método de calibração

A **calibração de câmera é obrigatória** (Requisito 3.A do manual) e reaproveita diretamente o método
validado pela equipe no **Laboratório 4**. O objetivo é obter os parâmetros que permitem **corrigir a
distorção** das lentes antes de qualquer análise, garantindo que medidas de área/posição das ROIs não
sejam deturpadas pela curvatura das bordas.

**Modelo.** A projeção segue o modelo *pinhole* $s\,[u\;v\;1]^\top = K\,[R\mid t]\,[X\;Y\;Z\;1]^\top$, com

$$K = \begin{bmatrix} f_x & 0 & c_x \\ 0 & f_y & c_y \\ 0 & 0 & 1 \end{bmatrix}, \qquad
\texttt{dist} = (k_1, k_2, p_1, p_2, k_3),$$

onde $K$ reúne os **intrínsecos** (distância focal e ponto principal), $[R\mid t]$ os **extrínsecos**
(pose) e `dist` os coeficientes de **distorção radial** ($k_i$) e **tangencial** ($p_i$).

**Procedimento.**
1. Imprimir o **tabuleiro de xadrez** (cantos internos conhecidos, p. ex. $6\times8$).
2. Capturar de **10 a 15 imagens** do tabuleiro em poses variadas com a câmera do sistema
   (programa `L4_chess.py` do Lab 4).
3. Detectar e refinar os cantos (`findChessboardCorners` + `cornerSubPix`) e estimar os parâmetros com
   `cv2.calibrateCamera`, obtendo `K`, `dist`, e os pares $(R,t)$ por imagem.
4. **Corrigir a distorção** de cada quadro em tempo real com `cv2.getOptimalNewCameraMatrix` +
   `cv2.undistort` (ou `cv2.remap`).

**Demonstração do erro de reprojeção (exigida pelo manual).** A qualidade da calibração é medida pelo
**erro de reprojeção**, a distância média (em pixels) entre os cantos detectados e os cantos
**reprojetados** com os parâmetros estimados:

$$\text{erro} = \frac{1}{N}\sum_{i=1}^{N} \big\lVert \, \mathbf{p}_i^{\text{obs}} - \pi(K, \texttt{dist}, R_i, t_i, \mathbf{P}_i) \, \big\rVert_2$$

calculada com `cv2.projectPoints` e `cv2.norm`; um RMS **abaixo de ~1 px** indica calibração adequada. O
valor obtido será reportado no Relatório Técnico (Etapa/Semana 11).


In [ ]:
# Método padrão para a DEMONSTRAÇÃO DO ERRO DE REPROJEÇÃO (a ser reportado com dados reais).
def erro_reprojecao(objpoints, imgpoints, rvecs, tvecs, K, dist):
    erro_total, n = 0.0, 0
    for i in range(len(objpoints)):
        proj, _ = cv2.projectPoints(objpoints[i], rvecs[i], tvecs[i], K, dist)
        erro = cv2.norm(imgpoints[i], proj, cv2.NORM_L2) / len(proj)
        erro_total += erro
        n += 1
    return erro_total / n   # erro medio de reprojeicao (pixels)


## 7. Método de avaliação funcional

A validação segue o item 4 do manual ("provar que o sistema é robusto") por meio de **métricas
objetivas**. Todas serão medidas na Etapa 4 (protótipo) e nos testes com voluntários (Etapas 5–7); aqui
definimos **como** cada uma será obtida.

**7.1 Desempenho — Latência / FPS.** Mede-se o tempo por quadro e a taxa média:
$\text{FPS} = 1 / \Delta t_{\text{quadro}}$. Meta: **≥ 15 FPS** e latência **≤ 66 ms/quadro** (RNF-01/02).

**7.2 Classificação — Matriz de confusão.** A tarefa é binária por pessoa/quadro: **CONFORME** (usa todos
os EPIs) × **NÃO CONFORME**. Comparando a saída do sistema com o *ground truth* anotado manualmente,
constrói-se a matriz:

|  | **Predito: Conforme** | **Predito: Não conforme** |
|---|---|---|
| **Real: Conforme** | VP | FN |
| **Real: Não conforme** | FP | VN |

E derivam-se as métricas:

$$\text{Precisão} = \frac{VP}{VP+FP}, \qquad \text{Revocação (Recall)} = \frac{VP}{VP+FN}, \qquad
F1 = 2\cdot\frac{\text{Precisão}\cdot\text{Revocação}}{\text{Precisão}+\text{Revocação}}$$

> No contexto de **segurança**, prioriza-se **alta revocação** para a classe "NÃO CONFORME" (é preferível
> um alarme falso a **deixar passar** alguém sem EPI). Esse critério guiará o ajuste dos limiares.

**7.3 Erro espacial (apoio).** Como a câmera é calibrada, pode-se reportar o **erro médio em pixels** na
localização das ROIs/pessoa, útil para diagnosticar falhas de enquadramento.

**7.4 Protocolo de teste com voluntários (Etapas 5–7).**
1. **Roteiro de tarefas:** o voluntário entra na área (a) com todos os EPIs, (b) sem capacete, (c) sem
   colete e (d) sem nenhum — cobrindo as quatro situações.
2. **Filmagem** (celular) da interação, conforme o manual.
3. **Anotação do *ground truth*** quadro a quadro e cálculo das métricas de 7.1–7.3.
4. **Coleta de feedback:** dificuldades, erros do sistema e sugestões de melhoria.

**7.5 Modelo de relato dos resultados (a preencher com dados reais).**

| Métrica | Meta | Valor obtido |
|---------|------|--------------|
| FPS médio | ≥ 15 | _a preencher_ |
| Latência média (ms) | ≤ 66 | _a preencher_ |
| Precisão | — | _a preencher_ |
| Revocação (NÃO CONFORME) | alta | _a preencher_ |
| F1-Score | — | _a preencher_ |
| Erro de reprojeção (calibração) | < 1 px | _a preencher_ |

> Os valores acima são **campos de preenchimento**: serão obtidos **experimentalmente** pela equipe. Nada
> nesta seção representa resultados medidos.


## 8. Cronograma e próximas etapas

Esta entrega corresponde à **Etapa 3 — Modelagem Funcional Geral**. As etapas subsequentes, conforme o
item 9 do manual, são:

| Etapa | Atividade | Situação |
|-------|-----------|----------|
| 1 | Entrevistas empáticas | Concluída (Etapa 2) |
| 2 | Contexto e cenário de aplicação | Concluída (Etapa 2) |
| **3** | **Modelagem funcional geral (este documento)** | **Entregue — 15/07/2026** |
| 4 | Desenvolvimento do projeto (hardware/software, protótipo) | A fazer |
| 5 | Elaboração do roteiro de teste com voluntários | A fazer |
| 6–7 | Seminários: testes com voluntários e análise de resultados | A fazer |
| 8 | Simpósio: artigo e apresentação | A fazer |


## 9. Declaração de uso de Inteligência Artificial Generativa

Em atendimento à **Portaria CNPq nº 2664/2026** (integridade na pesquisa quanto ao uso de Inteligência
Artificial Generativa — IAG), a equipe declara, de forma transparente:

- **Ferramenta utilizada:** assistente de IA generativa (modelo de linguagem da Anthropic — Claude).
- **Finalidade:** apoio à **organização e redação** deste documento de modelagem funcional (estruturação
  das seções, redação técnica, comentários do esboço de código e formatação do notebook).
- **Autoria e concepção:** a **definição do tema, o cenário de aplicação, os requisitos e as decisões de
  arquitetura** são da equipe, derivados das entrevistas empáticas e das aulas da disciplina. A IAG **não
  gerou dados experimentais** — nenhum resultado, métrica ou entrevista foi fabricado.
- **Responsabilidade:** a equipe é integralmente responsável pelo conteúdo final, tendo revisado e
  validado todas as informações aqui apresentadas.


## 10. Referências

1. TRUCCO, E.; VERRI, A. *Introductory Techniques for 3-D Computer Vision*. Prentice Hall, 1998.
2. OpenCV — *HOG (Histogram of Oriented Gradients) people detection*. Disponível em:
   https://docs.opencv.org/4.x/d5/d33/structcv_1_1HOGDescriptor.html
3. OpenCV — *Changing Colorspaces (segmentação HSV)*. Disponível em:
   https://docs.opencv.org/4.x/df/d9d/tutorial_py_colorspaces.html
4. OpenCV — *Camera Calibration*. Disponível em:
   https://docs.opencv.org/4.x/dc/dbb/tutorial_py_calibration.html
5. LearnOpenCV — *Camera Calibration using OpenCV*. Disponível em:
   https://learnopencv.com/camera-calibration-using-opencv/
6. OpenCV — *Object Tracking (Tracker API)*. Disponível em:
   https://docs.opencv.org/4.x/d9/df8/group__tracking.html
7. CNPq — *Portaria nº 2664/2026* (diretrizes de integridade e uso de IAG). Disponível em:
   http://memoria2.cnpq.br/web/guest/view/-/journal_content/56_INSTANCE_0oED/10157/23142775
